In [18]:
import scanpy as sc
import pandas as pd
from scipy import sparse
import anndata as ad

In [4]:
import os

In [25]:
adata_rna = sc.read_10x_h5(r"C:\Users\mniu\Documents\neuroblastoma-multiome\dataraw\GSM8159271_Be2c_filtered_feature_bc_matrix.h5")

C:\envs\nb\lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
C:\envs\nb\lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [5]:
print(adata_rna)
print(adata_rna.obs.head())
print(adata_rna.var.head())

AnnData object with n_obs × n_vars = 18473 × 36601
    var: 'gene_ids', 'feature_types', 'genome', 'interval'
Empty DataFrame
Columns: []
Index: [AAACAGCCAAACTAAG-1, AAACAGCCAAGCGAGC-1, AAACAGCCAATTGAGA-1, AAACAGCCACAAAGCG-1, AAACAGCCACCCACCT-1]
                    gene_ids    feature_types  genome            interval
MIR1302-2HG  ENSG00000243485  Gene Expression  GRCh38    chr1:29553-30267
FAM138A      ENSG00000237613  Gene Expression  GRCh38    chr1:36080-36081
OR4F5        ENSG00000186092  Gene Expression  GRCh38    chr1:65418-69055
AL627309.1   ENSG00000238009  Gene Expression  GRCh38  chr1:120931-133723
AL627309.3   ENSG00000239945  Gene Expression  GRCh38    chr1:91104-91105


In [ ]:
with open(r"C:\Users\mniu\Documents\neuroblastoma-multiome\dataraw\GSM8159270_count_matrix_Be2c_atac.csv") as f:
    for i in range(5):
        print(f.readline())

In [5]:
size_mb = os.path.getsize(r"C:\Users\mniu\Documents\neuroblastoma-multiome\dataraw\GSM8159270_count_matrix_Be2c_atac.csv") / (1024*1024)
print(f"{size_mb:.1f} MB")

4824.6 MB


In [6]:
atac_preview = pd.read_csv(r"C:\Users\mniu\Documents\neuroblastoma-multiome\dataraw\GSM8159270_count_matrix_Be2c_atac.csv", sep = ";", index_col=0,
                     nrows=100)

In [17]:
print(atac_preview.shape)
print(atac_preview.head())

(100, 11695)
                    AAACAGCCAAACTAAG.1  AAACAGCCAATTGAGA.1  \
chr1-10037-10397                     0                   0   
chr1-180556-181606                   0                   0   
chr1-184071-184508                   0                   0   
chr1-190825-191618                   0                   0   
chr1-777657-779308                   0                   0   

                    AAACAGCCACAAAGCG.1  AAACAGCCACCTACGG.1  \
chr1-10037-10397                     0                   0   
chr1-180556-181606                   0                   0   
chr1-184071-184508                   0                   0   
chr1-190825-191618                   1                   2   
chr1-777657-779308                   0                   1   

                    AAACATGCAGCGCTTG.1  AAACATGCAGGATTAA.1  \
chr1-10037-10397                     0                   0   
chr1-180556-181606                   0                   0   
chr1-184071-184508                   1                 

In [19]:
filepath = atac_df = r"C:\Users\mniu\Documents\neuroblastoma-multiome\dataraw\GSM8159270_count_matrix_Be2c_atac.csv"
chunksize = 5000
sparse_chunks = []
peak_names = []
reader = pd.read_csv(filepath, sep=";", index_col = 0, chunksize = chunksize)

In [20]:
for i, chunk in enumerate(reader):
    sparse_chunk = sparse.csr_matrix(chunk.values)
    sparse_chunks.append(sparse_chunk)
    peak_names.extend(chunk.index.tolist())
    if i == 0:
        cell_barcodes = chunk.columns.tolist()
    print(f"Processed chunk {i+1}, total rows so far: {len(peak_names)}")

full_matrix = sparse.vstack(sparse_chunks)
print(full_matrix.shape)

Processed chunk 1, total rows so far: 5000
Processed chunk 2, total rows so far: 10000
Processed chunk 3, total rows so far: 15000
Processed chunk 4, total rows so far: 20000
Processed chunk 5, total rows so far: 25000
Processed chunk 6, total rows so far: 30000
Processed chunk 7, total rows so far: 35000
Processed chunk 8, total rows so far: 40000
Processed chunk 9, total rows so far: 45000
Processed chunk 10, total rows so far: 50000
Processed chunk 11, total rows so far: 55000
Processed chunk 12, total rows so far: 60000
Processed chunk 13, total rows so far: 65000
Processed chunk 14, total rows so far: 70000
Processed chunk 15, total rows so far: 75000
Processed chunk 16, total rows so far: 80000
Processed chunk 17, total rows so far: 85000
Processed chunk 18, total rows so far: 90000
Processed chunk 19, total rows so far: 95000
Processed chunk 20, total rows so far: 100000
Processed chunk 21, total rows so far: 105000
Processed chunk 22, total rows so far: 110000
Processed chunk 2

In [22]:
adata_atac = ad.AnnData(X = full_matrix.T.tocsr(), obs = pd.DataFrame(index = cell_barcodes), var = pd.DataFrame(index = peak_names))
print(adata_atac)

AnnData object with n_obs × n_vars = 11695 × 216035


In [33]:
adata_atac.write_h5ad("dataraw/GSM8159270_Be2c_atac.h5ad")

In [26]:
rna_barcodes = set(adata_rna.obs_names)
atac_barcodes = set(adata_atac.obs_names)

In [27]:
print("RNA barcode example:", list(rna_barcodes)[:3])
print("ATAC barcode example:", list(atac_barcodes)[:3])
print("Overlap:", len(rna_barcodes & atac_barcodes))
print("RNA total:", len(rna_barcodes), "| ATAC total:", len(atac_barcodes))

RNA barcode example: ['CTGATCACAGTTAAAG-1', 'CTCCTCACAACTCGCG-1', 'GACGCAACAAGTAAGC-1']
ATAC barcode example: ['ATTTGTGAGTTAGCCG.1', 'ATTTGTGAGCTCCTTA.1', 'ATCCTTAGTAAAGCGG.1']
Overlap: 0
RNA total: 18473 | ATAC total: 11695


In [28]:
adata_atac.obs_names = adata_atac.obs_names.str.replace(".", "-", regex = False)

In [29]:
atac_barcodes = set(adata_atac.obs_names)
print("Overlap:", len(rna_barcodes & atac_barcodes))

Overlap: 11695


In [32]:
gitignore_content = """dataraw/
models/
*.h5
*.h5ad
*.csv
neuroblastoma-env/
.ipynb_checkpoints/
"""

with open(".gitignore", "w") as f:
    f.write(gitignore_content)

In [34]:
!pip freeze > requirements.txt